# GPT-SoVITS Improve — Fast Full Install (Kaggle)

이 노트북은 최신 `Colab-Inference-Fast.ipynb`의 흐름을 **그대로 유지하면서 Kaggle에 필요한 부분만 최소 수정**한 버전입니다.

변경점은 아래 정도입니다.

- Google Drive 마운트 제거
- Kaggle Dataset의 `training.list` + `datas/` 또는 이를 담은 압축파일을 한 셀에서 지정
- `/content/...` 작업 경로를 `/kaggle/working/...`으로 변경
- WebUI의 기본 학습 데이터 경로를 Kaggle 경로로 자동 입력

Kaggle Notebook 설정에서 **GPU**와 **Internet**을 켜고 위에서부터 실행하세요.


## 0. Kaggle 데이터셋 경로

Kaggle에 업로드한 Dataset 안에 다음 둘이 들어 있으면 됩니다.

```text
training.list
datas/
```

또는 둘을 하나의 `.zip`, `.tar`, `.tar.gz`, `.tgz` 파일로 묶어 올려도 됩니다.

대부분은 `KAGGLE_DATASET_ROOT = None` 그대로 두면 자동으로 찾습니다.  
여러 Dataset을 붙였다면 `/kaggle/input/<dataset-slug>` 경로를 직접 지정하세요.


In [ ]:
from pathlib import Path
import shutil
import tarfile
import zipfile

# 여러 Kaggle Dataset을 붙인 경우에만 직접 지정하세요.
# 예: Path("/kaggle/input/my-gpt-sovits-dataset")
KAGGLE_DATASET_ROOT = None

KAGGLE_INPUT = Path("/kaggle/input")
RUNTIME_DATA_ROOT = Path("/kaggle/working/audio_input")
RUNTIME_DATA_ROOT.mkdir(parents=True, exist_ok=True)


def find_dataset_source(root: Path):
    """training.list+datas 또는 이를 포함한 압축파일을 찾습니다."""
    direct_lists = list(root.rglob("training.list"))
    for list_path in direct_lists:
        data_dir = list_path.parent / "datas"
        if data_dir.is_dir():
            return ("direct", list_path.parent)

    archives = []
    for pattern in ("*.zip", "*.tar", "*.tar.gz", "*.tgz"):
        archives.extend(root.rglob(pattern))
    if archives:
        return ("archive", archives[0])

    raise FileNotFoundError(
        "Kaggle Dataset에서 training.list + datas/ 또는 압축파일을 찾지 못했습니다."
    )


search_root = Path(KAGGLE_DATASET_ROOT) if KAGGLE_DATASET_ROOT else KAGGLE_INPUT
mode, source = find_dataset_source(search_root)

if mode == "direct":
    SOURCE_ROOT = source
else:
    extract_root = RUNTIME_DATA_ROOT / "extracted"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    print("[dataset] 압축 해제:", source)
    if source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            zf.extractall(extract_root)
    else:
        with tarfile.open(source, "r:*") as tf:
            tf.extractall(extract_root)

    _, SOURCE_ROOT = find_dataset_source(extract_root)

SOURCE_LIST = SOURCE_ROOT / "training.list"
SOURCE_DATAS = SOURCE_ROOT / "datas"

if not SOURCE_LIST.is_file():
    raise FileNotFoundError(SOURCE_LIST)
if not SOURCE_DATAS.is_dir():
    raise FileNotFoundError(SOURCE_DATAS)

# Kaggle input은 읽기 전용이므로 list만 working 쪽에 새로 만듭니다.
TRAINING_LIST = RUNTIME_DATA_ROOT / "training.list"
DATAS_DIR = SOURCE_DATAS

# Colab/Windows/기존 절대경로가 들어 있더라도 첫 필드의 파일명 기준으로
# Kaggle의 실제 datas/ 경로로 바꿉니다.
raw_lines = SOURCE_LIST.read_text(encoding="utf-8-sig").splitlines()
fixed_lines = []
missing = []

for line in raw_lines:
    if not line.strip():
        continue

    parts = line.split("|")
    if len(parts) < 4:
        raise ValueError(f"잘못된 training.list 행: {line}")

    old_audio = Path(parts[0].replace("\\", "/"))
    candidate = DATAS_DIR / old_audio.name

    if not candidate.is_file():
        # 혹시 datas 하위 폴더 구조가 있는 경우 basename으로 재탐색
        matches = list(DATAS_DIR.rglob(old_audio.name))
        if len(matches) == 1:
            candidate = matches[0]
        else:
            missing.append(old_audio.name)

    parts[0] = str(candidate)
    fixed_lines.append("|".join(parts))

if missing:
    preview = ", ".join(missing[:10])
    raise FileNotFoundError(
        f"training.list에 있지만 datas/에서 찾지 못한 파일이 {len(missing)}개 있습니다: {preview}"
    )

TRAINING_LIST.write_text("\n".join(fixed_lines) + "\n", encoding="utf-8")

print("✅ Kaggle dataset 준비 완료")
print("TRAINING_LIST =", TRAINING_LIST)
print("DATAS_DIR     =", DATAS_DIR)
print("samples       =", len(fixed_lines))


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time


REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_DIR = Path("/kaggle/working/GPT-SoVITS")


print("[1/2] GPT-SoVITS-improve 최신 main 준비", flush=True)

start = time.perf_counter()

if (REPO_DIR / ".git").exists():
    print("[git] 기존 repo 유지 + 최신 main 동기화", flush=True)

    subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", "main"],
        cwd=REPO_DIR,
        check=True,
    )

    subprocess.run(
        ["git", "reset", "--hard", "origin/main"],
        cwd=REPO_DIR,
        check=True,
    )

else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    print("[git] 새 repo shallow clone", flush=True)

    clone_cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "--single-branch",
        "--branch",
        "main",
        REPO_URL,
        str(REPO_DIR),
    ]

    subprocess.run(clone_cmd, check=True)


elapsed = time.perf_counter() - start
print(f"[1/2] repo 준비 완료: {elapsed:.1f}s", flush=True)


print("[2/2] 전체 의존성 + pretrained 설치 시작", flush=True)

setup_script = REPO_DIR / "tools" / "colab_fast_setup.py"

# upstream Colab 설치 로직은 그대로 사용하고,
# Colab 전용 작업 경로만 Kaggle working 경로로 최소 치환합니다.
setup_text = setup_script.read_text(encoding="utf-8")
setup_text = setup_text.replace("/content/", "/kaggle/working/")
setup_script.write_text(setup_text, encoding="utf-8")


print("[check] setup script 문법 검사", flush=True)

subprocess.run(
    [sys.executable, "-m", "py_compile", str(setup_script)],
    check=True,
)


print("[run] setup script 실행 — 최근 로그만 표시", flush=True)

from collections import deque
from IPython.display import clear_output

setup_log = Path("/kaggle/working/gpt_sovits_setup.log")
tail = deque(maxlen=24)
pending = ""
last_render = 0.0


def render_tail(final=False):
    global last_render
    now = time.monotonic()
    if not final and now - last_render < 0.8:
        return
    last_render = now
    clear_output(wait=True)
    print("[1/2] repo 준비 완료")
    print("[2/2] 전체 의존성 + pretrained 설치")
    print(f"[log] 전체 로그: {setup_log}")
    print("-" * 72)
    if tail:
        print("\n".join(tail))
    else:
        print("설치 프로세스 시작 중...")


with setup_log.open("w", encoding="utf-8") as log_writer:
    process = subprocess.Popen(
        [sys.executable, "-u", str(setup_script)],
        stdout=log_writer,
        stderr=subprocess.STDOUT,
        text=True,
    )

    with setup_log.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as log_reader:
        while process.poll() is None:
            chunk = log_reader.read()
            if chunk:
                pending += chunk.replace("\r", "\n")
                parts = pending.split("\n")
                pending = parts.pop()
                for item in parts:
                    item = item.strip()
                    if item:
                        tail.append(item)
                render_tail()
            time.sleep(0.25)

        chunk = log_reader.read()
        if chunk:
            pending += chunk.replace("\r", "\n")

if pending:
    for item in pending.split("\n"):
        item = item.strip()
        if item:
            tail.append(item)

return_code = process.wait()
render_tail(final=True)

if return_code != 0:
    error_tail = setup_log.read_text(
        encoding="utf-8",
        errors="replace",
    ).replace("\r", "\n").splitlines()[-80:]
    clear_output(wait=True)
    print("\n".join(error_tail))
    raise RuntimeError(
        f"setup script failed with exit code {return_code}. "
        f"전체 로그: {setup_log}"
    )

print("\n[OK] 설치 완료", flush=True)


## V5dev 전체 WebUI — 한국어

Colab판과 동일한 전체 WebUI를 실행합니다.

- 한국어 `ko_KR`
- 기본 버전 `V5dev`
- 전처리 `a → b → c → d`
- GPT / SoVITS 학습
- emotion controller
- 추론

위 데이터셋 셀에서 잡힌 `training.list`와 `datas/`가 WebUI 기본값으로 자동 입력됩니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


REPO_DIR = Path("/kaggle/working/GPT-SoVITS")

# 최신 upstream WebUI 구조는 그대로 두고,
# Colab 기본 데이터 경로 두 곳만 이번 Kaggle Dataset 경로로 바꿉니다.
webui_path = REPO_DIR / "webui.py"
webui_text = webui_path.read_text(encoding="utf-8")

webui_text = webui_text.replace(
    "/content/drive/MyDrive/audio_input/training.list",
    str(TRAINING_LIST),
)
webui_text = webui_text.replace(
    "/content/drive/MyDrive/audio_input/datas",
    str(DATAS_DIR),
)

webui_path.write_text(webui_text, encoding="utf-8")

# 수정 후 문법 확인
subprocess.run(
    [sys.executable, "-m", "py_compile", str(webui_path)],
    check=True,
)


env = os.environ.copy()

python_path = [
    str(REPO_DIR),
    str(REPO_DIR / "GPT_SoVITS"),
]

existing_python_path = env.get("PYTHONPATH")

if existing_python_path:
    python_path.append(existing_python_path)


env.update(
    {
        "PYTHONPATH": os.pathsep.join(python_path),
        "language": "ko_KR",
        "version": "v5dev",
        "is_share": "True",
        "HF_HOME": "/kaggle/working/huggingface",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "HF_HUB_DISABLE_UPDATE_CHECK": "1",
        "HF_HUB_DOWNLOAD_TIMEOUT": "120",
        "HF_XET_HIGH_PERFORMANCE": "1",
    }
)


webui_cmd = [
    sys.executable,
    "-u",
    "webui.py",
    "ko_KR",
]


print("[dataset] training.list:", TRAINING_LIST)
print("[dataset] datas        :", DATAS_DIR)
print("[run] 파인튜닝 WebUI 시작 — 한국어", flush=True)

process = subprocess.Popen(
    webui_cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"webui.py failed with exit code {return_code}. "
        "위에 표시된 마지막 traceback/에러 메시지를 확인하세요."
    )
